# Student Details

**Name:** Shahbaz Ali  
**Roll No.:** 2401830007  
**Course:** B.Sc. Cyber Security  
**Semester:** 5th

---

# Lab 4: Explainable and Privacy-Aware AI Threat Alert Lab

This notebook demonstrates baseline predictions, interpretability via feature importances, controlled sensitivity testing, privacy masking, and responsible incident alert generation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

features = ['failed_logins', 'login_hour', 'new_device', 'download_mb']

X_train = pd.DataFrame([
    {'failed_logins': 0, 'login_hour': 10, 'new_device': 0, 'download_mb': 40, 'label': 'Normal'},
    {'failed_logins': 7, 'login_hour': 2, 'new_device': 1, 'download_mb': 150, 'label': 'Suspicious'},
    {'failed_logins': 1, 'login_hour': 15, 'new_device': 0, 'download_mb': 720, 'label': 'Normal'},
    {'failed_logins': 4, 'login_hour': 23, 'new_device': 1, 'download_mb': 400, 'label': 'Suspicious'},
    {'failed_logins': 2, 'login_hour': 9, 'new_device': 0, 'download_mb': 80, 'label': 'Normal'},
    {'failed_logins': 0, 'login_hour': 12, 'new_device': 0, 'download_mb': 20, 'label': 'Normal'},
    {'failed_logins': 6, 'login_hour': 1, 'new_device': 1, 'download_mb': 300, 'label': 'Suspicious'}
])

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train[features], X_train['label'])
print('Model trained and ready!')

Model trained and ready!


## Q1. Baseline Predictions with Confidence

We run predictions on five baseline sanitized records.

In [2]:
samples = pd.DataFrame([
    {'account_id': 'ACC-1001', 'ip_address': '192.168.10.24', 'failed_logins': 0, 'login_hour': 10, 'new_device': 0, 'download_mb': 40},
    {'account_id': 'ACC-1002', 'ip_address': '10.0.4.18', 'failed_logins': 7, 'login_hour': 2, 'new_device': 1, 'download_mb': 150},
    {'account_id': 'ACC-1003', 'ip_address': '172.16.8.90', 'failed_logins': 1, 'login_hour': 15, 'new_device': 0, 'download_mb': 720},
    {'account_id': 'ACC-1004', 'ip_address': '192.168.20.11', 'failed_logins': 4, 'login_hour': 23, 'new_device': 1, 'download_mb': 400},
    {'account_id': 'ACC-1005', 'ip_address': '10.1.1.9', 'failed_logins': 2, 'login_hour': 9, 'new_device': 0, 'download_mb': 80}
])

samples['prediction'] = model.predict(samples[features])
samples['confidence'] = model.predict_proba(samples[features]).max(axis=1).round(3)
display(samples)

account_id    ip_address      failed_logins  login_hour  new_device  download_mb  prediction  confidence
ACC-1001      192.168.10.24   0              10          0           40           Normal      1.0
ACC-1002      10.0.4.18       7              2           1           150          Suspicious  1.0
ACC-1003      172.16.8.90     1              15          0           720          Normal      1.0
ACC-1004      192.168.20.11   4              23          1           400          Suspicious  1.0
ACC-1005      10.1.1.9        2              9           0           80           Normal      1.0

## Q2. Feature Importance and Explanation

We visualize and display feature importance mappings.

In [3]:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance)

plt.figure(figsize=(6, 4))
plt.bar(importance['feature'], importance['importance'], color='teal')
plt.title('Feature Importance for Login Alert')
plt.ylabel('Importance')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

for feature in importance.head(3)['feature']:
    print(feature, 'was frequently used by the model to separate records.')

feature        importance
failed_logins  1.0
login_hour     0.0
new_device     0.0
download_mb    0.0

failed_logins was frequently used by the model to separate records.
login_hour was frequently used by the model to separate records.
new_device was frequently used by the model to separate records.


## Q3. Controlled Sensitivity Test

We alter a numeric feature on a record and compare the predicted label.

In [4]:
baseline = samples.iloc[[4]].copy()
changed = baseline.copy()
changed['failed_logins'] = 8

baseline_prediction = model.predict(baseline[features])[0]
baseline_confidence = model.predict_proba(baseline[features]).max()
changed_prediction = model.predict(changed[features])[0]
changed_confidence = model.predict_proba(changed[features]).max()

print('Baseline:', baseline_prediction, '| Confidence:', baseline_confidence)
print('Changed :', changed_prediction, '| Confidence:', changed_confidence)

Baseline: Normal | Confidence: 1.0
Changed : Suspicious | Confidence: 1.0


## Q4. Privacy Masking and Responsible-Use Rules

We mask IP addresses and account identifiers before displaying the results.

In [5]:
def mask_ip(ip):
    parts = str(ip).split('.')
    if len(parts) == 4:
        return f'{parts[0]}.{parts[1]}.xxx.xxx'
    return '<MASKED_IP>'

def mask_account(account_id):
    text = str(account_id)
    return text[:4] + '***' + text[-2:]

samples['masked_ip'] = samples['ip_address'].apply(mask_ip)
samples['masked_account'] = samples['account_id'].apply(mask_account)

display(samples[[
    'masked_account', 'masked_ip',
    'prediction', 'confidence'
]])

privacy_rules = [
    'Mask identifiers before display or export.',
    'Use sanitized data in classroom notebooks.',
    'Restrict access to unmasked records.',
    'Require human review for high-risk action.'
]

for rule in privacy_rules:
    print('-', rule)

masked_account  masked_ip         prediction  confidence
ACC-***01       192.168.xxx.xxx   Normal      1.0
ACC-***02       10.0.xxx.xxx      Suspicious  1.0
ACC-***03       172.16.xxx.xxx    Normal      1.0
ACC-***04       192.168.xxx.xxx   Suspicious  1.0
ACC-***05       10.1.xxx.xxx      Normal      1.0

- Mask identifiers before display or export.
- Use sanitized data in classroom notebooks.
- Restrict access to unmasked records.
- Require human review for high-risk action.


## Q5. Responsible Alert Generation

We generate an alert containing severity, reason, recommended action and human-review status.

In [6]:
def numeric_reason(row):
    reasons = []
    if row['failed_logins'] >= 5:
        reasons.append('many failed logins')
    if row['new_device'] == 1:
        reasons.append('new device')
    if row['login_hour'] < 6 or row['login_hour'] > 22:
        reasons.append('unusual login time')
    if row['download_mb'] >= 500:
        reasons.append('large data download')
    return ', '.join(reasons) if reasons else 'no strong warning'

def make_alert(row):
    reason = numeric_reason(row)

    if row['prediction'] == 'Suspicious':
        severity = 'High'
        action = 'Verify, contain if required, and report.'
    elif row['confidence'] < 0.70:
        severity = 'Medium'
        action = 'Send for analyst review before action.'
    else:
        severity = 'Low'
        action = 'Continue normal monitoring.'

    human_review = 'Yes' if severity in ['High', 'Medium'] else 'No'
    return severity, reason, action, human_review

alerts = samples.apply(make_alert, axis=1)
samples[['severity', 'reason', 'recommended_action', 'human_review']] = pd.DataFrame(alerts.tolist(), index=samples.index)

display(samples[[
    'masked_account', 'prediction', 'confidence',
    'severity', 'reason',
    'recommended_action', 'human_review'
]])

masked_account  prediction  confidence  severity  reason                                      recommended_action                          human_review
ACC-***01       Normal      1.0         Low       no strong warning                           Continue normal monitoring.                No
ACC-***02       Suspicious  1.0         High      many failed logins, new device, unusual login time  Verify, contain if required, and report.  Yes
ACC-***03       Normal      1.0         Low       large data download                         Continue normal monitoring.                No
ACC-***04       Suspicious  1.0         High      new device, unusual login time             Verify, contain if required, and report.  Yes
ACC-***05       Normal      1.0         Low       no strong warning                           Continue normal monitoring.                No

## Conclusion

The lab demonstrates how a small AI-based login-activity model can produce explainable and privacy-aware security alerts. Confidence scores, feature importance, controlled sensitivity testing, identifier masking, and human-review guidance help make the alerting workflow more transparent and responsible.